# Ten task probes → native steering → transcoder overlap

Choose **A100 / high RAM**, then **Run all**. This is a separate pilot; it preserves the running old-vs-new comparison and quick-box data. A separate Colab runtime avoids competing with an active job for GPU memory. Setup reuses local downloads and the comparison environment when available.

**What we test:** ten separately fitted linear probes, one for each LIBERO spatial task's two actual bowl-instance instructions. Examples include cookie-box/cabinet, stove/cabinet, and ramekin/nearby-bowl choices. The notebook resolves instructions from the benchmark BDDL rather than inventing spatial labels. These are related tasks, sometimes reversing a similar contrast, **not ten independent concepts**. Each probe asks which bowl instruction was supplied at the same starting scene; this can reflect wording as well as intended target.

1. **Discovery: starts 0–3.** Capture native layer-8 MLP outputs at denoising call index 4 (the fifth of ten calls). Average across action tokens to give one vector per prompt/start. Fit L2-regularized logistic regression (`C=0.1`) and a mean-contrast direction. Four paired starts are a deliberately small pilot, not a reliable generalization estimate. No simulated rollout is needed for discovery.
2. **Validation: starts 4–5.** Keep the alternate instruction and initial pixels fixed. Compare native probe, mean-contrast, and one random direction at a 10% output RMS budget. Freeze whichever of probe/mean-contrast gives the higher mean movement score. Mark validation as supported only if both starts have positive native prompt controls and positive selected-direction effects, and its mean exceeds the random control. Failed cases are still reported and tested, not quietly removed.
3. **Map the frozen direction into both decoders.** Greedy orthogonal matching pursuit fits 4, 16, and 64 decoder columns. Report approximation error and cosine, with five random-column subsets as a geometric reference. Use **16 features**, fixed in advance, for steering. No test activations or outcomes select directions, features, or counts. Compare old/new columns by one-to-one absolute cosine alignment; feature IDs are local to a dictionary. A geometric match is not proof of semantic identity.
4. **Test: fresh starts 10–12.** Native alternate/target instructions, zero edit, full-budget native direction, norm-matched native/old/new directions, two random native directions, and a full target-activation patch. The patch uses the paired target-prompt output at this exact layer/call; it is a localization control with a different budget, not a competing sparse method. Steering keeps the alternate instruction and initial scene unchanged.

**Equal edit sizes:** both sparse code offsets are clipped against their current nonnegative activations. All five matched arms (native, old, new, two random) are then shrunk to the smallest feasible output RMS, including BF16 rounding. This state-dependent calibration uses no outcome. It can reduce the shared budget substantially, even to zero. The report checks repeated calibration across conditions, zero-edit equality, and a ≤2% realized-norm spread. A zero feasible intervention is flagged as unavailable. Read `dense_full` separately: it checks native steering at the requested budget. Clipping-induced direction changes are recorded.

**Overlap has three meanings here:** decoder approximation versus feature count, alignment of selected old/new decoder columns and their test activation contrasts, and preservation of held-out movement effects. None alone establishes a causal circuit. Reconstruction of native outputs at the selected layer/time is also saved. This does not repeat the all-layer replacement study.

The movement score is `50 × (edited − alternate-baseline)` final relative bowl preference, where preference is `(distance to alternate bowl − distance to target bowl) / bowl separation`, using initial bowl positions. Positive means movement toward the target relative to the baseline. **Not success percentage.** Only the first 50-action chunk is executed; no replan or full-episode success estimate. Probe test starts are excluded from fitting but may have appeared in transcoder pretraining. Three test starts/task and two random directions cannot establish significance. No p-values are produced.

**Speed and resume:** defaults use 480 predictions (80 discovery-only captures, 400 short simulator replays), not 480 full episodes. One frozen BF16 teacher and both selected-layer dictionaries remain on GPU. OMP dictionary correlations use GPU; small least-squares fits use CPU. No gradients or expensive circuit tracing. Every finished trial and fit is checksummed and committed to Drive. With `RESUME_RUN_ID="latest"`, rerun after interruption to skip compatible completed work; only the unfinished trial restarts. Changed settings or source create a separate run. Per-task `summaries/taskXX/summary.json` is available before all ten finish. Errors stop immediately rather than repeating the same failure across ten tasks.


In [ ]:
# @title Ten-case pilot controls
from pathlib import Path
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908" # @param {type:"string"}
OLD_CHECKPOINT = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"
NEW_CHECKPOINT = "outputs/trajectory-training/pi05-trajectory-transcoders-001/trained_transcoders.pt"
NEW_SHA256 = "ea582a91af113f20c232a545ebc11f47c5e2d6f115929ae7a3a96ccd0f74784e"
CHECKPOINTS = {"old": OLD_CHECKPOINT, "new": NEW_CHECKPOINT}
POLICY_PATH = "lerobot/pi05_libero_finetuned"
SAVE_RUNS_TO_DRIVE = True
MIN_GPU_MEMORY_GB = 20
RESUME_RUN_ID = "latest" # @param {type:"string"}
TASKS = "0,1,2,3,4,5,6,7,8,9" # @param {type:"string"}
TRAIN_STARTS = "0,1,2,3" # @param {type:"string"}
VALIDATION_STARTS = "4,5" # @param {type:"string"}
TEST_STARTS = "10,11,12" # @param {type:"string"}
LAYER = 8 # @param {type:"integer"}
DENOISING_CALL = 4 # @param {type:"integer"}
FEATURE_COUNTS = "4,16,64" # @param {type:"string"}
STEERING_FEATURES = 16 # @param {type:"integer"}
RMS_BUDGET = 0.1 # @param {type:"number"}
SEED = 1000
parse = lambda s: list(map(int,s.split(',')))
N_TASKS = len(parse(TASKS))
CALLS = N_TASKS*(2*len(parse(TRAIN_STARTS))+5*len(parse(VALIDATION_STARTS))+10*len(parse(TEST_STARTS)))
print(f"{N_TASKS} separate task probes; {CALLS} single-chunk predictions total (480 at defaults).")
print("Discovery captures do not run the simulator forward. Validation/test use 50 steps.")
print("One teacher load, two selected-layer dictionaries resident on GPU; all completed trials cached to Drive.")


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())

# Everything this notebook measures is written under DRIVE_ROOT/outputs, because
# Colab's disk is wiped when the runtime ends. If the folder is not actually
# there, copytree would happily create a fresh unshared directory and the run
# would look saved while being invisible to anyone else. Fail here instead.
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
if SAVE_RUNS_TO_DRIVE:
    if not DRIVE_ROOT.exists():
        raise RuntimeError(
            f"{DRIVE_ROOT} does not exist under the mounted Drive.\n"
            "Check DRIVE_ROOT in Controls against the shared folder, and make sure the "
            "folder is added to *your* My Drive (Shared with me is not mounted).\n"
            "Set SAVE_RUNS_TO_DRIVE = False only if you accept losing the run."
        )
    DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_OUTPUTS / ".write_test"
    probe.write_text("ok")          # a read-only share fails here, not four hours in
    assert probe.read_text() == "ok"
    probe.unlink()
    free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
    print(f"Drive writable. Free on the mount: {free_gb:.1f} GB")
    print(f"Everything this notebook produces lands under {DRIVE_OUTPUTS}")
else:
    print("SAVE_RUNS_TO_DRIVE is off: this run will be lost when the runtime ends.")


In [ ]:
# @title Install / Repair Runtime (rerun safely after an interrupted install)
import os, subprocess, shlex
from collections import deque
from pathlib import Path

VENV = Path('/content/pi05-comparison-venv')
PYTHON = VENV / 'bin/python'
UV_BIN_DIR = Path('/content/uv-bin')
UV = str(UV_BIN_DIR / 'uv')
INSTALL_LOG = Path('/content/pi05-comparison-install.log')
FORK_COMMIT = 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'

def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print('$', shlex.join(cmd), flush=True)
    tail = deque(maxlen=100)
    with INSTALL_LOG.open('a') as log:
        log.write('\n$ ' + shlex.join(cmd) + '\n')
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush(); tail.append(line)
                status = proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired: proc.kill(); proc.wait()
                raise
    if status:
        raise RuntimeError(f'Command failed (exit {status}); log: {INSTALL_LOG}\n' + ''.join(tail))
    return subprocess.CompletedProcess(cmd, status)

os.environ['PATH'] = f'{VENV / "bin"}:{UV_BIN_DIR}:' + os.environ['PATH']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'

# A Python executable alone does not mean a previous install succeeded.
CHECK = r"""
import json, importlib.metadata as md, importlib.util
expected = {'lerobot':'0.4.4','torch':'2.10.0','torchvision':'0.25.0',
            'hf-libero':'0.1.4','numpy':'2.4.6','matplotlib':'3.10.0','av':'15.1.0'}
for name, value in expected.items():
    assert md.version(name).split('+')[0] == value, (name, md.version(name), value)
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert origin.get('vcs_info',{}).get('commit_id') == 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'
from transformers.models.siglip import check
assert check.check_whether_transformers_replace_is_installed_correctly()
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
import torch, matplotlib, av, numpy
assert torch.cuda.is_available(), 'CUDA unavailable in comparison Python'
assert all(importlib.util.find_spec(n) for n in ('libero','egl_probe'))
print('Validated Pi0.5 fork, simulation imports, CUDA, and versions:', expected)
"""
ready = False
if PYTHON.exists():
    probe = subprocess.run([str(PYTHON), '-c', CHECK], capture_output=True, text=True)
    ready = probe.returncode == 0
    print(probe.stdout if ready else 'Incomplete/incompatible runtime; repairing.\n' + probe.stderr[-2500:])
if not ready:
    apt_env = {**os.environ, 'DEBIAN_FRONTEND':'noninteractive'}
    run(['apt-get','update','-qq'], env=apt_env)
    run(['apt-get','install','-y','-qq','build-essential','ca-certificates','cmake',
         'curl','ffmpeg','git','pkg-config','libegl1','libgl1','libglib2.0-0',
         'libglvnd0','libglx0','libopengl0','libosmesa6-dev','libsm6','libxext6','libxrender1'],env=apt_env)
    if not Path(UV).exists():
        UV_BIN_DIR.mkdir(parents=True,exist_ok=True)
        run(['curl','-LsSf','https://astral.sh/uv/install.sh','-o','/tmp/install-uv.sh'])
        run(['sh','/tmp/install-uv.sh'], env={**os.environ,'UV_INSTALL_DIR':str(UV_BIN_DIR)})
    if not PYTHON.exists():
        run([UV,'python','install','3.12'])
        run([UV,'venv',str(VENV),'--python','3.12'])
    # Match training: egl-probe requires pre-CMake-4 build tooling.
    run([UV,'pip','install','--python',PYTHON,'cmake==3.31.6','setuptools>=71,<81','wheel'])
    run([UV,'pip','install','--python',PYTHON,'--no-build-isolation-package','egl-probe','egl-probe==1.0.2'])
    override = Path('/content/pi05-comparison-transformers-overrides.txt')
    override.write_text(f'transformers @ git+https://github.com/huggingface/transformers.git@{FORK_COMMIT}\n')
    run([UV,'pip','install','--python',PYTHON,'--torch-backend','cu128',
         '--overrides',override,'--no-build-isolation-package','egl-probe','cmake==3.31.6',
         'lerobot[libero]==0.4.4','torch==2.10.0','torchvision==0.25.0',
         'hf-libero==0.1.4','hf-transfer','opencv-python','numpy==2.4.6',
         'matplotlib==3.10.0','av==15.1.0','scikit-learn','joblib','pillow'])
run([PYTHON,'-c',CHECK])
print('Runtime ready. All model stages run in this separate Python; no kernel restart needed.')


In [ ]:
# @title Restore the bundled source (no GitHub update required)
import base64, hashlib, json, os, subprocess, zlib
from pathlib import Path
SOURCE_SHA256 = '231868ab88efc277284e14bf0cafca409b40e3d2854add7d5cd8d6a5937226d8'
RUNNER_SHA256 = 'f1dd77f0d1c311a86667b084ec643638438abf7b696983907a97b41ab26cb4ae'
SOURCE_BASE_COMMIT = '79ccf32a10df551a0d33167d43e03cee85241f18'
payload = '...'
raw = zlib.decompress(base64.b64decode(payload))
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256
LOCAL_REPO = Path("/content/pi05-probe-overlap-source") / SOURCE_SHA256[:12]
for name, contents in json.loads(raw).items():
    target = LOCAL_REPO / name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(contents)
os.environ["PYTHONPATH"] = os.pathsep.join([str(LOCAL_REPO/'src'),str(LOCAL_REPO/'scripts')])
print("Source:",LOCAL_REPO)


In [ ]:
# @title Check evaluation imports before running either model
preflight_code = """
import importlib, sys
from pi05_mi.lerobot_runtime import patch_transformers_causal_mask_compat, patch_pi05_checkpoint_key_compat
patch_transformers_causal_mask_compat()
patch_pi05_checkpoint_key_compat()
for name in ('run_pi05_probe_overlap', 'probe_pi05_transcoder_counterfactual', 'compare_transcoder_fidelity',
             'collect_pi05_transcoder_features', 'trace_pi05_transcoder_circuit'):
    importlib.import_module(name)
    print('Import OK:', name, flush=True)
assert 'train_pi05_transcoders' not in sys.modules
import torch, numpy as np
from probe_pi05_transcoder_counterfactual import LatentAccumulator
gpu=torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB', flush=True)
# Check the optimized transport on the actual GPU before reusing old results.
reference=LatentAccumulator(defer_transfers=False)
fast=LatentAccumulator()
reference.enabled=fast.enabled=True
values=torch.arange(1024, device='cuda', dtype=torch.float32).reshape(1,8,128).to(torch.bfloat16)
for step in range(3):
    reference.callback('test',0,values+step,torch.tensor([0.5],device='cuda'))
    fast.callback('test',0,values+step,torch.tensor([0.5],device='cuda'))
fast.flush()
for key in reference.mean:
    np.testing.assert_array_equal(reference.mean[key],fast.mean[key])
    np.testing.assert_array_equal(reference.max[key],fast.max[key])
print('Batched GPU statistics transfer verified; experiment settings unchanged.', flush=True)
print('Evaluation import preflight passed.', flush=True)
"""
run([PYTHON, '-u', '-c', preflight_code])

# Exercise the new norm matching on the actual GPU before any simulator work.
matching_check = """
import torch, numpy as np
from pi05_mi.probe_overlap import matched_edits
torch.manual_seed(71)
y=torch.randn(1,50,128,device='cuda').to(torch.bfloat16)
weights={a:torch.randn(128,32,device='cuda') for a in ('old','new')}
codes={a:torch.rand(1,50,32,device='cuda')*.02 for a in weights}
maps={a:dict(indices=[0,1],coefficients=[1.,-.5]) for a in weights}
outputs,info=matched_edits(y,codes,weights,maps,np.ones(128),[np.arange(128),np.arange(128)[::-1]],.1)
assert info['norm_match_ok'], info
print('GPU BF16 feasible norm-matching check passed:',info,flush=True)
"""
run([PYTHON,'-u','-c',matching_check])


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    print("Downloading LIBERO assets (cached files are reused)...", flush=True)
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-u", "-c", assets_code])

# Freeze the teacher to the exact revision recorded by the new training run.
teacher_result = LOCAL_REPO / 'teacher_path.txt'
run([PYTHON, '-u', LOCAL_REPO / 'scripts/download_pi05_comparison_teacher.py',
     '--result-path', teacher_result])
POLICY_PATH = teacher_result.read_text().strip()
print('Pinned teacher:', POLICY_PATH)
# Parse the actual checkpoint config before launching any evaluation tasks.
run([PYTHON, '-u', '-c',
     "from argparse import Namespace; from pi05_mi.lerobot_runtime import _load_train_config; "
     "import sys; cfg=_load_train_config(Namespace(policy_path=sys.argv[1],local_files_only=True)); "
     "print('Teacher config OK:', type(cfg.policy).__name__)", POLICY_PATH])


In [ ]:
# @title Verify and stage both checkpoints
import hashlib, shutil
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(16*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
CHECKPOINT_HASHES, LOCAL_CHECKPOINTS = {}, {}
for arm,relative in CHECKPOINTS.items():
    source=DRIVE_ROOT/relative
    if not source.is_file(): raise FileNotFoundError(source)
    local=Path('/content/pi05-comparison-checkpoints')/arm/source.name
    local.parent.mkdir(parents=True,exist_ok=True)
    print('Hashing',arm,'checkpoint...',flush=True)
    expected=sha256(source)
    if arm=='new' and expected!=NEW_SHA256:
        raise ValueError('New checkpoint does not match the completed, downloaded training bundle')
    if not local.exists() or sha256(local)!=expected:
        partial=local.with_suffix('.partial');shutil.copyfile(source,partial)
        if sha256(partial)!=expected: raise IOError('Checkpoint copy hash mismatch')
        partial.replace(local)
    CHECKPOINT_HASHES[arm]=expected;LOCAL_CHECKPOINTS[arm]=str(local)
    print(arm,local,expected)
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['old']])
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['new']])


In [ ]:
# @title Restore a compatible run
import sys
sys.path.insert(0,str(LOCAL_REPO/'scripts'))
from comparison_resume import ResumeCache,select_run,atomic_json
settings=dict(tasks=parse(TASKS),train=parse(TRAIN_STARTS),validation=parse(VALIDATION_STARTS),test=parse(TEST_STARTS),
    layer=LAYER,call_index=DENOISING_CALL,ks=parse(FEATURE_COUNTS),steering_k=STEERING_FEATURES,budget=RMS_BUDGET,seed=SEED)
identity=dict(protocol="native-probe-overlap-v1",source=SOURCE_SHA256,runner=RUNNER_SHA256,
    checkpoints=CHECKPOINT_HASHES,teacher_revision="8e174154ef5f6c60a8da12ae99c303d8963138c1",
    settings=settings,inference_steps=10,action_steps=50,policy_dtype="bfloat16")
local_parent=LOCAL_REPO/'outputs/probe-overlap'
drive_parent=DRIVE_ROOT/'outputs/probe-overlap' if SAVE_RUNS_TO_DRIVE else None
RUN_ID=select_run(local_parent,drive_parent,identity,RESUME_RUN_ID)
OVERLAP_ROOT=local_parent/RUN_ID
OVERLAP_DRIVE=drive_parent/RUN_ID if drive_parent else None
CACHE=ResumeCache(OVERLAP_ROOT,OVERLAP_DRIVE,LOCAL_REPO)
(OVERLAP_ROOT/'source_snapshot.json').write_bytes(raw)
atomic_json(OVERLAP_ROOT/'source_manifest.json',dict(sha256=SOURCE_SHA256,base_commit=SOURCE_BASE_COMMIT))
CACHE.complete('source',CACHE.signature(['source']),[OVERLAP_ROOT/'source_snapshot.json',OVERLAP_ROOT/'source_manifest.json'])
print('Run:',RUN_ID,'\nResults:',OVERLAP_DRIVE or OVERLAP_ROOT)


In [ ]:
# @title Run the ten probes and steering comparisons
cmd=[PYTHON,'-u',LOCAL_REPO/'scripts/run_pi05_probe_overlap.py',
    '--old-checkpoint',LOCAL_CHECKPOINTS['old'],'--new-checkpoint',LOCAL_CHECKPOINTS['new'],
    '--policy-path',POLICY_PATH,'--resume-root',OVERLAP_ROOT,'--tasks',TASKS,
    '--train',TRAIN_STARTS,'--validation',VALIDATION_STARTS,'--test',TEST_STARTS,
    '--layer',LAYER,'--call-index',DENOISING_CALL,'--ks',FEATURE_COUNTS,
    '--steering-k',STEERING_FEATURES,'--budget',RMS_BUDGET,'--seed',SEED]
if OVERLAP_DRIVE:cmd+=['--drive-root',OVERLAP_DRIVE]
run(cmd)


In [ ]:
# @title Read results and control failures
import json
from IPython.display import display,Image
report=json.loads((OVERLAP_ROOT/'overlap_report.json').read_text())
for case in report['cases']:
    print('\nTASK',case['task'],case['prompts'])
    print('Selected native direction:',case['frozen']['selected'])
    print('Validation supported:',case['frozen']['validation_supported'],'Test probe:',case['probe_test'])
    print('Valid matched starts:',case['valid_matched_starts'],'/',len(case['test']))
    print('Mean movement:',{k:round(v,4) for k,v in case['mean_scores'].items()})
    print('Valid matched means:',case['valid_matched_mean_scores'])
    print('Retained native effect ratios (only when matched native mean is positive):',case['retained_effect_ratio'])
    for arm in ('old','new'):
        print(arm,'direction approximation errors:',{k:round(v['relative_error'],4) for k,v in case['mapping']['fits'][arm].items()})
display(Image(filename=str(OVERLAP_ROOT/'figures/overlap.png')))
print('CSV tables and PNG:',(OVERLAP_DRIVE or OVERLAP_ROOT)/'figures')
print('All tasks are reported. Examine controls and realized norms before comparing steering scores.')
print('No significance claim: three test starts/task and two random directions are exploratory.')


In [ ]:
# @title Watch a test start (change task and start to inspect another case)
PREVIEW_TASK = 0 # @param {type:"integer"}
PREVIEW_START = 10 # @param {type:"integer"}
from IPython.display import display,Image,HTML
import html
for condition in ('native_-1','native_1','dense_full','dense','old','new','random_0','patch'):
    path=OVERLAP_ROOT/'cases'/f'task{PREVIEW_TASK:02d}_init{PREVIEW_START:02d}'/condition/'preview.gif'
    if path.exists():
        display(HTML('<b>'+html.escape(condition)+'</b>'))
        display(Image(filename=str(path)))
print('First predicted chunk only; GIF playback is illustrative.')
